# 🔬 Deep Researcher

A multi-agent pipeline that:
1. **Plans** — decides which web searches to run
2. **Searches** — runs them in parallel
3. **Writes** — synthesises a 1 000+ word Markdown report
4. **Emails** — delivers the report via SendGrid (optional)

---

## 1  Setup

In [ ]:
# Install dependencies if needed
# %pip install openai-agents sendgrid python-dotenv pydantic

In [ ]:
from utils.runner import run_pipeline
from utils.display import display_report

## 2  Configuration

Edit `config/settings.py` (or set environment variables in `.env`) to configure:

| Variable | Description |
|---|---|
| `OPENAI_API_KEY` | Your OpenAI API key |
| `SENDGRID_API_KEY` | Your SendGrid API key |
| `FROM_EMAIL` | Sender address (must be verified in SendGrid) |
| `TO_EMAIL` | Recipient address |
| `HOW_MANY_SEARCHES` | Number of parallel searches (default 3) |

## 3  Run a quick single search (sanity check)

> ⚠️ Each call to `WebSearchTool` costs ~$0.025 on OpenAI.

In [ ]:
from agents import Agent, WebSearchTool, Runner, trace
from agents.model_settings import ModelSettings
from IPython.display import display, Markdown

_quick_agent = Agent(
    name="QuickSearch",
    instructions="Search the web and return a 2-3 paragraph summary. No extra commentary.",
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

with trace("QuickSearch"):
    result = await Runner.run(_quick_agent, "Latest AI Agent frameworks in 2025")

display(Markdown(result.final_output))

## 4  Run the full pipeline

In [ ]:
QUERY = "Latest AI Agent frameworks in 2025"

# Set send_email=False if you haven't configured SendGrid yet
report = await run_pipeline(query=QUERY, send_email=False)

In [ ]:
display_report(report)

## 5  Inspect structured output directly

In [ ]:
print("Short summary:\n", report.short_summary)
print("\nFollow-up questions:")
for q in report.follow_up_questions:
    print(f"  • {q}")